# YTAgent — Interactive Free Colab Film Renderer

This notebook renders **inside your own interactive Colab session**. It does not connect to Railway, claim queue jobs, run a background worker, or require API keys. Free GPU availability is not guaranteed. Google may stop intensive content-generation sessions, so this is a best-effort route, not an always-on service.

It uses Wan 2.1 1.3B for ten moving scenes, Kokoro for local narration, and FFmpeg for one complete vertical MP4 with burned-in captions. It needs a GPU runtime and can take a long time.

**Start:** Runtime → Change runtime type → GPU. Run the cells from top to bottom. Edit the storyboard cell before rendering if you want a different film. The included storyboard is a sample about deep-ocean pressure.

**Privacy:** No worker token, Hugging Face token, Railway secret, or paid API key is needed. Model downloads are public. Output files live in the temporary Colab runtime until downloaded.

In [ ]:
# Install runtime dependencies. Keep Colab's preinstalled CUDA-enabled PyTorch.
import os, subprocess, sys, pathlib, json, textwrap, time
def run(cmd):
    print('RUN:', ' '.join(map(str, cmd)))
    subprocess.run(cmd, check=True)
run(['apt-get','-qq','update'])
run(['apt-get','-qq','install','-y','ffmpeg','espeak-ng','fonts-dejavu-core'])
run([sys.executable,'-m','pip','install','-q','diffusers>=0.32.0','transformers>=4.45.0','accelerate>=1.0.0','ftfy','imageio[ffmpeg]','soundfile>=0.12.1','fastapi>=0.115.0','pydantic>=2.0','spacy>=3.8.2,<4','num2words','phonemizer-fork','espeakng-loader','loguru','addict'])
run([sys.executable,'-m','pip','install','-q','--no-deps','git+https://github.com/hexgrad/misaki.git'])
run([sys.executable,'-m','pip','install','-q','--no-deps','git+https://github.com/hexgrad/kokoro.git'])
print('Dependency setup complete.')

In [ ]:
# GPU preflight. Stop here if Colab did not assign a CUDA GPU.
import torch, subprocess
print(subprocess.run(['nvidia-smi'],capture_output=True,text=True).stdout)
if not torch.cuda.is_available():
    raise RuntimeError('No GPU is attached. Select Runtime > Change runtime type > GPU, reconnect, and rerun this cell.')
print('CUDA:', torch.version.cuda, '| GPU:', torch.cuda.get_device_name(0), '| VRAM GiB:', round(torch.cuda.get_device_properties(0).total_memory/1024**3,1))

## 1. Film storyboard

Keep exactly ten scenes. Each narration line should be short enough for a six-second scene. Visual prompts must describe visible physical actions, not title cards or static slides. Replace the sample text to create another film.

In [ ]:
TOPIC = 'Why the deepest part of the ocean can crush a submarine'
SEED = 20261115
STORYBOARD = [
 {'prompt':'Cinematic documentary 3D realism, a modern research submarine descends through dark blue open ocean, tiny bioluminescent particles drift past its floodlights, camera follows from behind, realistic water movement, no text.', 'narration':'At the ocean’s deepest point, the pressure can crush an ordinary submarine in seconds.', 'caption':'At the deepest point, pressure can crush an ordinary submarine.'},
 {'prompt':'A research vessel on a vast rolling ocean at dawn lowers a compact deep sea submersible on a heavy cable, crew silhouettes work carefully, wide cinematic shot, natural waves and moving clouds, no text.', 'narration':'The journey begins on the surface, where pressure feels almost exactly like normal air.', 'caption':'The journey begins where pressure feels familiar.'},
 {'prompt':'Underwater tracking shot beside a yellow research submersible descending past a sunlit layer into deep cobalt water, beams fade gradually, suspended marine snow drifts, realistic hull details, no text.', 'narration':'But every ten metres of seawater adds another atmosphere of crushing pressure.', 'caption':'Every ten metres adds another atmosphere of pressure.'},
 {'prompt':'Close cinematic view of a deep sea submersible pressure sphere as water darkens around it, tiny bubbles and particles stream upward, camera slowly pushes toward thick metal hull, realistic lighting, no text.', 'narration':'By one thousand metres, the water pushes with roughly one hundred times surface pressure.', 'caption':'At one thousand metres, pressure is roughly one hundred times greater.'},
 {'prompt':'Inside a deep ocean engineering workshop, gloved hands inspect a thick titanium pressure sphere and measure its curved surface with precision tools, warm practical lights, shallow depth of field, no text.', 'narration':'Engineers use thick rounded pressure spheres because curves spread the force more evenly.', 'caption':'Rounded pressure spheres spread the force more evenly.'},
 {'prompt':'A detailed underwater view shows a deep diving submersible descending past a giant dark rock wall, its compact spherical cabin remains steady while water currents move around it, dramatic tracking camera, no text.', 'narration':'A sphere has no flat wall that can easily buckle inward under the load.', 'caption':'A sphere resists inward buckling better than a flat wall.'},
 {'prompt':'Inside a submarine control cabin, a pilot watches depth instruments and checks a glowing mechanical gauge, subtle instrument reflections on the window, steady handheld documentary camera, realistic human movement, no text.', 'narration':'Pilots constantly monitor depth because even tiny structural weaknesses can become dangerous.', 'caption':'Pilots monitor depth because tiny weaknesses can become dangerous.'},
 {'prompt':'A dramatic but realistic deep ocean shot of a damaged test pressure vessel imploding underwater in a brief cloud of bubbles and fragments, distant camera, no people, scientifically grounded, no gore, no text.', 'narration':'If the hull fails, surrounding water rushes inward with extraordinary destructive force.', 'caption':'If the hull fails, surrounding water rushes inward.'},
 {'prompt':'A remotely operated deep sea vehicle explores the ocean floor beside strange pale organisms and black volcanic rocks, robotic arms move gently through drifting sediment, cinematic documentary realism, no text.', 'narration':'That is why only specially engineered vehicles can safely explore extreme ocean depths.', 'caption':'Only specially engineered vehicles can explore extreme depths.'},
 {'prompt':'Final majestic wide shot of a tiny research submersible hovering above the deepest ocean trench, powerful lights reveal a vast dark abyss, camera slowly pulls back, realistic cinematic documentary, no text.', 'narration':'The ocean is beautiful, but its greatest depths demand extraordinary engineering and respect.', 'caption':'The deep ocean demands extraordinary engineering and respect.'}
]
assert len(STORYBOARD) == 10, 'Exactly ten scenes are required.'
assert all(len(x['narration'].split()) <= 16 for x in STORYBOARD), 'Shorten narration lines to 16 words maximum.'
print('Ready:', TOPIC, '| scenes:', len(STORYBOARD))

In [ ]:
# 2. Generate ten moving scenes locally with the existing YTAgent Wan worker code.
REPO = '/content/YTAgent'
if not os.path.isdir(REPO): run(['git','clone','--depth','1','https://github.com/qingremzy/YTAgent.git',REPO])
else: run(['git','-C',REPO,'pull','--ff-only','origin','main'])
os.environ['YT_WORKER_DIR'] = '/content/ytagent-render'
os.environ['WAN_NUM_FRAMES'] = '49'
os.environ['WAN_STEPS'] = '6'
os.environ['WAN_FPS'] = '8'
os.environ['HF_HUB_DISABLE_TELEMETRY'] = '1'
sys.path.insert(0, REPO + '/gpu-worker')
import app as wan_worker
prompts = [x['prompt'] for x in STORYBOARD]
local_job_id = 'colab-' + str(int(time.time()))
wan_worker.JOBS[local_job_id] = {'status':'queued','progress':0,'message':'Interactive Colab render'}
print('Starting Wan 2.1 scene generation. This may take a long time on a free T4.')
wan_worker.run_job(local_job_id, prompts, SEED, remote_job_id=None)
result = wan_worker.JOBS[local_job_id]
print(result)
if result.get('status') != 'completed': raise RuntimeError(result.get('message','Wan generation failed.'))
WORK = pathlib.Path(os.environ['YT_WORKER_DIR']) / local_job_id
scene_files = sorted(WORK.glob('scene-*.mp4'))
assert len(scene_files) == 10, f'Expected 10 scene clips, found {len(scene_files)}'
print('Generated scene clips:', len(scene_files))

In [ ]:
# 3. Generate natural local narration and align each line to its scene.
import numpy as np, soundfile as sf
audio_files=[]
for i, scene in enumerate(STORYBOARD):
    wav = WORK / f'narration-{i:02d}.wav'
    wan_worker.generate_kokoro_audio(scene['narration'], wav)
    audio_files.append(wav)
    print(f'Narration {i+1}/10 ready')
sample_rate=24000
slot_seconds=6.0
audio_parts=[]
for wav in audio_files:
    data, sr = sf.read(str(wav), dtype='float32')
    if data.ndim > 1: data=data.mean(axis=1)
    if sr != sample_rate: raise RuntimeError(f'Unexpected audio sample rate: {sr}')
    target=int(slot_seconds*sample_rate)
    data=data[:target]
    if len(data)<target: data=np.pad(data,(0,target-len(data)))
    audio_parts.append(data)
narration_track=np.concatenate(audio_parts)
narration_path=WORK/'full-narration.wav'
sf.write(str(narration_path),narration_track,sample_rate,subtype='PCM_16')
print('Narration duration:', round(len(narration_track)/sample_rate,2),'seconds')

In [ ]:
# 4. Compile one vertical MP4 with burned-in captions, then inspect its streams and duration.
def srt_time(seconds):
    ms=round(seconds*1000); h,ms=divmod(ms,3600000); m,ms=divmod(ms,60000); s,ms=divmod(ms,1000)
    return f'{h:02}:{m:02}:{s:02},{ms:03}'
srt_path=WORK/'captions.srt'
with open(srt_path,'w',encoding='utf-8') as f:
    for i,scene in enumerate(STORYBOARD):
        f.write(f'{i+1}\n{srt_time(i*slot_seconds)} --> {srt_time((i+1)*slot_seconds-0.08)}\n{scene["caption"]}\n\n')
concat_path=WORK/'scenes.txt'
with open(concat_path,'w') as f:
    for clip in scene_files: f.write("file '"+str(clip)+"'\n")
silent_video=WORK/'picture.mp4'
vf=f"fps=30,scale=1080:1920:force_original_aspect_ratio=increase,crop=1080:1920,subtitles=filename='{srt_path}':force_style='FontName=DejaVu Sans,FontSize=20,PrimaryColour=&H00FFFFFF,OutlineColour=&H00000000,BorderStyle=1,Outline=2,Shadow=1,Alignment=2,MarginV=110'"
run(['ffmpeg','-y','-f','concat','-safe','0','-i',str(concat_path),'-t','60','-vf',vf,'-an','-c:v','libx264','-preset','veryfast','-crf','20','-pix_fmt','yuv420p',str(silent_video)])
final_path=pathlib.Path('/content/YTAgent-Film.mp4')
run(['ffmpeg','-y','-i',str(silent_video),'-i',str(narration_path),'-t','60','-map','0:v:0','-map','1:a:0','-c:v','copy','-c:a','aac','-b:a','160k','-movflags','+faststart',str(final_path)])
probe=subprocess.run(['ffprobe','-v','error','-show_entries','format=duration,size:stream=codec_type,width,height','-of','json',str(final_path)],capture_output=True,text=True,check=True)
info=json.loads(probe.stdout)
print(json.dumps(info,indent=2))
duration=float(info['format']['duration'])
assert 58 <= duration <= 61, f'Unexpected final duration: {duration}'
assert any(x.get('codec_type')=='video' for x in info['streams']), 'Missing video stream'
assert any(x.get('codec_type')=='audio' for x in info['streams']), 'Missing audio stream'
print('MP4 passed basic stream and duration checks. Play it before publishing.')

In [ ]:
# 5. Download the finished MP4 to your device.
from google.colab import files
files.download(str(final_path))